#Phase 1: Draft Stage
This notebook produces codes and justifications from a structured dataframe. Inference is performed locally at one interview excerpt (two Q-A pairs) at a time.

In [ ]:
#Import libraries
import os
import torch
from pydantic import BaseModel, Field
from typing import List

import outlines
import transformers
import torch


from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

### Read Data

In [ ]:
import pandas as pd
import json

df = pd.read_parquet(OUTPUT_DIR / "test"/"test_interviews.parquet") #segmented dataframe with interview segments and identifiers

df.columns



### Load LLM model

initialize token and LLM model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "mistralai/Mistral-7B-Instruct-v0.3" # You need a HF token

tokenizer = AutoTokenizer.from_pretrained(model_id) #Mistral's tokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" #A100 or H100
)

print("Mistral-7B-Instruct-v0.3 is loaded.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Mistral-7B-Instruct-v0.3 is loaded.


### LLM configurations with Outlines

In [ ]:
from transformers import GenerationConfig

model.generation_config = GenerationConfig.from_model_config(model.config)

model.generation_config.max_new_tokens = 400 #max token outout
model.generation_config.min_new_tokens = 40 #min new tokens
model.generation_config.temperature = 0.0 # Temp set to 0.0 because it is overridden by Outlines in structured output
model.generation_config.do_sample = False #LLM chooses most likely next token = False
model.generation_config.pad_token_id = tokenizer.eos_token_id # Mistral does not have a pad token, so we pad with EOS
model.generation_config.eos_token_id = tokenizer.eos_token_id #Mistral's end-of-sequence token

### Import Outlines for validation of output

In [ ]:
import outlines

outlines_model = outlines.from_transformers(model, tokenizer)

# Draft stage - preparation of inference

### Format draft stage prompt in INST (chat_template)

In [ ]:

def format_prompt(content, tokenizer): #Taking analysis prompt and sending it to model in instrct mode
    messages = [
        {"role": "user", "content": content}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
#Draft stage prompt - on segments (interview_excerpt)

def get_draft_stage_prompt(interview_excerpt):

    content = f"""
    ### TASK
    You are an expert qualitative researcher. Your task is to perform Thematic Analysis (Braun & Clarke 2006)
    on the following interview excerpts.

    ### CONTEXT
    The research objective is to understand the interviewees’ individual experiences of the profound transformations in Danish society
    and family life from the 1960s-1980s, characterized by changing gender roles, women’s increasing labour market participation,
    the weakening of traditional marriage, and the expansion of family policies.

    ### GUIDELINES
    - Max 1-3 codes per interview excerpt. Each code must have a clear, concise name in one shorthand phrase.
    - Provide a brief justification for each code that describes the analytical rationale for each code.
    - Interview_excerpt include both interviewer questions “Q: ” and informant answers “A: ”.
    - “Q: “ is to be used for context only which means that the codes should be based mainly on informant responses.
    - Interview_excerpts that ONLY provide demographic information (e.g., age, gender) must be coded as "not relevant".

    ### INTERVIEW EXCERPT TO ANALYZE
    {interview_excerpt}

    ### INSTRUCTIONS FOR RESPONSE
    - Return only the required structured data.
    - Do not include placeholders such as "none".
    - Do not include any explanatory text outside the structured output.
    - Each code must contain a valid code label and justification.

    ### FINAL INSTRUCTION
    Analyze the interview_excerpt.
    """
    return content

# Defining output schema for capturing structured output with Pydantic
-schemas are used during inference, parsing and for validation of outputs


In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List

class Code(BaseModel):
    code: str = Field(
        min_length=1,
        description="Clear, concise name of code in shorthand language or a short sentence"
    )
    justification: str = Field(
        min_length=1,
        description="1-2 sentences that describe the analytical rationale and abstraction for the coding"
    )

    @field_validator("code", "justification")
    @classmethod
    def no_empty_or_whitespace(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("Field cannot be empty or whitespace only")
        return v

    @field_validator("code")
    @classmethod
    def no_placeholder_codes(cls, v: str) -> str:
        blocked = {"none", "null", "n/a", "na", "n_a", "unknown"}
        if v.strip().lower() in blocked:
            raise ValueError(f"Invalid placeholder code: {v}")
        return v

class CodingResponse(BaseModel):
    codes: List[Code] = Field(
        min_length=1,
        max_length=3,
        description="A list of relevant codes for the interview segment (typically 1-3 codes). If the excerpt is not analytically relevant, return exactly one code with code='not_relevant'. Do not return any additional codes in that case."
    )

# Main function for analyzing in draft stage, parsing and validating with Outlines

- Builds an analysis prompt from the interview excerpt and formats it using Mistral’s instruction (INST) template.
- Sends the formatted prompt to the language model via Outlines by enforcing a structured response schema (`CodingResponse`).
- Retry logic: Attempts the generation multiple times in case of failure, with a delay between retries.
- Error handling: capture and handle errors if the model fails to return valid structured output

In [ ]:
import json
import time

def analyze_excerpt(interview_excerpt, max_retries=3, retry_delay=0.2):
    content = get_draft_stage_prompt(interview_excerpt)
    prompt_inst = format_prompt(content, tokenizer) #Prompt is delivered in INST wrapper
    last_error = None

    for attempt in range(max_retries):
        try:
            result = outlines_model(prompt_inst, CodingResponse)


            parsed = json.loads(result) #format JSON str to dict
            validated = CodingResponse(**parsed) #unpacking dict


            return validated

        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    raise ValueError(f"Failed after {max_retries} attempts: {last_error}")

# Validation and normalization
-used for validation of output after inference and parsing

In [ ]:
import re

#Function for reg.ex normalization of codes
def normalize_code_regex_only(code: str) -> str:
    if code is None:
        return None

    code = code.strip()


    # Transform camelCase to underscores
    code = re.sub(r'(?<!^)(?=[A-Z])', '_', code)

    # lowercase
    code = code.lower()

    #Know error for misspelling of relevant
    code = code.replace("relevent", "relevant")

    # replace whitespace and hyphens with underscore
    code = re.sub(r"[\s\-]+", "_", code)

    # remove all other tokens than letters, numbers and underscore
    code = re.sub(r"[^a-z0-9_]", "", code)

    # Collapes multiple underscores
    code = re.sub(r"_+", "_", code)

    # remove underscores in start and end of code
    code = code.strip("_")

    return code

#  MAIN: Function for processing each row and picking up errors - loop
-call all functions together

In [ ]:
def process_row(row):
    try:
        validated = analyze_excerpt(row["interview_excerpt"]) #analyse with prompt and validate output

        codes = [normalize_code_regex_only(item.code) for item in validated.codes] #normalize
        justifications = [item.justification.strip() for item in validated.codes] #normalize

        return {
            "unit_id": int(row["unit_id"]),
            "codes": codes,
            "justifications": justifications,
            "status": "ok",
            "error": None,

        }

    except Exception as e:
        return {
            "unit_id": int(row["unit_id"]),
            "codes": None,
            "justifications": None,
            "status": "failed",
            "error": str(e),
            "excerpt": row["interview_excerpt"]
        }

# Run pipeline per interview per row for local analysis and inference

In [ ]:
import time
import pandas as pd

unique_int_ids = df["int_id"].unique()
total_interviews = len(unique_int_ids)

print(f"Starting thematic coding draft phase... ({total_interviews} interviews)\n")

start_total = time.time()

results = []


for i, int_id in enumerate(unique_int_ids, start=1): #
    interview_start = time.time() #Timing the process

    print(f"\n--- Interview {i}/{total_interviews}: {int_id} ---")

    df_current = df[df["int_id"] == int_id] #analyse per interview per row for local context

    print(f"Number of excerpts: {len(df_current)}")

    for _, row in df_current.iterrows():
        results.append(process_row(row))

    interview_time = time.time() - interview_start
    print(f"Finished interview {int_id} in {round(interview_time, 2)} sec") #Timing the process

total_time = time.time() - start_total

results_df = pd.DataFrame(results)

print(f"\nTotal time: {round(total_time, 2)} sec ({round(total_time/60, 2)} min)")

### Saving results

In [ ]:
from datetime import datetime
import shutil

# Keep only columns with result from analysis
results_to_merge = results_df[ ["unit_id", "codes", "justifications", "status", "error"] ].copy()

# merge results on copy of original df
df_results_draft = df.copy().merge(
    results_to_merge,
    on="unit_id",
    how="left"
)

In [ ]:


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save as PARQUET (source of truth for the next step of self_refine flow)
timestamp_path_parquet = DRAFT_DIR / f"test_run_{timestamp}.parquet"
current_path_parquet = DRAFT_DIR / "currentTEST.parquet"

df_results_draft.to_parquet(timestamp_path_parquet, index=False)
shutil.copy(timestamp_path_parquet, current_path_parquet)

